# Temperature Control

{class}`~pylabrobot.capabilities.temperature_controlling.temperature_controller.TemperatureController` provides heating and (optionally) active cooling. It is one of the most widely used capabilities -- found on heater-shakers, incubators, plate readers, and standalone thermal modules.

Temperature controllers are machines that can heat and/or actively cool a material or enclosed volume from a room-temperature baseline (~20--25 °C). Multi-functional machines like heater-shakers, qPCR instruments, and smart storage systems build on top of this capability.

## Actuation technologies

Multiple technologies can be used to implement temperature control:

- **Thermoelectric (Peltier) modules** -- solid-state devices that pump heat via the Peltier effect, enabling both heating and cooling by reversing current flow. Compact and modular, they mount directly on robotic decks. *Pros:* bidirectional control, fast response, minimal footprint. *Cons:* limited ΔT from ambient (~±65 °C max), efficiency drops near extremes.

- **Liquid-circulation systems** -- external chillers or heaters pump fluid (water or glycol) through channels around a sample block, delivering uniform, stable temperatures well below and above ambient. *Pros:* broad temperature range, excellent uniformity. *Cons:* bulky, requires plumbing.

## When to use

Use this capability whenever you need to hold labware at a specific temperature: incubation at 37 °C, enzyme inactivation at 65 °C, keeping reagents cold at 4 °C, etc.

## Walkthrough

In this example we use a chatterbox (simulated) backend. On real hardware, the capability is accessed as an attribute on a device (e.g. `hs.tc` on a Hamilton Heater Shaker).

In [ ]:
from pylabrobot.capabilities.temperature_controlling import TemperatureController
from pylabrobot.capabilities.temperature_controlling.chatterbox import TemperatureControllerChatterboxBackend

tc = TemperatureController(backend=TemperatureControllerChatterboxBackend())
await tc._on_setup()

Set a target temperature and wait for it to stabilize. `set_temperature` sends the command and returns immediately. `wait_for_temperature` polls every 1 second until the target is reached (within `tolerance`).

In [ ]:
await tc.set_temperature(37.0)
await tc.wait_for_temperature(tolerance=0.5)

Read the current temperature at any time:

In [ ]:
current = await tc.request_temperature()
print(f"{current:.1f} °C")

Deactivate to stop heating/cooling and return to ambient. This resets `target_temperature` to `None`.

In [ ]:
await tc.deactivate()

## Tips and gotchas

- **`set_temperature` does not wait.** It sends the command and returns. Use `wait_for_temperature` to block until the target is reached.
- **Passive cooling.** If you set a target below the current temperature and the backend doesn't support active cooling, you'll get a `ValueError`. Pass `passive=True` to allow the device to cool naturally.
- **`wait_for_temperature` polls every 1 second.** It raises `TimeoutError` after `timeout` seconds (default 300) and `RuntimeError` if no target has been set.

## Supported hardware

```{supported-devices} heating, cooling
```

## API reference

See {class}`~pylabrobot.capabilities.temperature_controlling.temperature_controller.TemperatureController` and {class}`~pylabrobot.capabilities.temperature_controlling.temperature_controller.TemperatureControllerBackend`.